In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [2]:
i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.

In [3]:
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\corn"

In [4]:
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    

Found 2690 files belonging to 3 classes.
Using 2152 files for training.


In [5]:
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set

Found 2690 files belonging to 3 classes.
Using 538 files for validation.


In [6]:
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)

In [7]:
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())

Classes: ['Corn___Common_Rust', 'Corn___Gray_Leaf_Spot', 'Corn___Northern_Leaf_Blight']
Train batches: 68
Val batches: 9
Test batches: 8


In [8]:
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)

In [9]:
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [10]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers


In [11]:
num = len(class_names)


In [12]:
base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False

In [13]:
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint

In [14]:
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

In [15]:
x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)

In [16]:
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)

In [17]:
model = models.Model(inputs, outputs)

In [18]:
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]

In [20]:
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6717 - loss: 0.8210

68/68 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - accuracy: 0.7941 - loss: 0.4989 - val_accuracy: 0.9113 - val_loss: 0.3138 - learning_rate: 0.0010
Epoch 2/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8709 - loss: 0.3107

68/68 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.8917 - loss: 0.2778 - val_accuracy: 0.9326 - val_loss: 0.2199 - learning_rate: 0.0010
Epoch 3/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9197 - loss: 0.2032

68/68 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.9140 - loss: 0.2168 - val_accuracy: 0.9149 - val_loss: 0.1913 - learning_rate: 0.0010
Epoch 4/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.9219 - loss: 0.1955 - val_accuracy: 0.9149 - val_loss: 0.2000 - learning_rate: 0.0010
Epoch 5/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.9233 - loss: 0.1924 - val_accuracy: 0.9113 - val_loss: 0.2185 - learning_rate: 0.0010
Epoch 6/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 956ms/step - accuracy: 0.9185 - loss: 0.2088

68/68 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.9257 - loss: 0.1855 - val_accuracy: 0.9255 - val_loss: 0.1692 - learning_rate: 3.0000e-04
Epoch 7/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.9289 - loss: 0.1760 - val_accuracy: 0.9113 - val_loss: 0.2285 - learning_rate: 3.0000e-04
Epoch 8/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.9368 - loss: 0.1658 - val_accuracy: 0.9362 - val_loss: 0.2049 - learning_rate: 3.0000e-04
Epoch 9/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.9270 - loss: 0.1819 - val_accuracy: 0.9255 - val_loss: 0.2122 - learning_rate: 9.0000e-05


In [21]:
model.save("corn.keras")

In [22]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9297 - loss: 0.1454
Test Loss: 0.1454
Test Accuracy: 92.97%
